# Inference and Evaluation — BBox-Indicator Visual Baseline (Vietnamese)

This notebook loads an exported model, generates predictions on the test split, and computes the final evaluation metrics:

- BLEU-4
- METEOR
- ROUGE-L
- CIDEr
- RefSigLIPScore-style score with SigLIP2
- CHAIR-S and CHAIR-I

The training notebook logs only loss and BLEU. The heavier metrics are computed here after inference.

## 0. Install dependencies

In [1]:
!pip install -q "transformers==4.51.3" "peft==0.15.2" "accelerate==1.6.0"
!pip install -q sentencepiece sacrebleu rouge-score nltk pycocoevalcap huggingface_hub pandas tqdm Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 167.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 129.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 23.7 MB/s eta 0:00:00


## 1. Imports and Google Drive mount

In [2]:
import os
import sys
import json
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

Mounted at /content/drive
torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-40GB


## 2. Paths and package import

In [3]:
THESIS_DIR = Path('/content/drive/MyDrive/[Thesis] Improved_transformer')
PROJECT_DIR = THESIS_DIR / 'bbox_indicator_baseline'

DATA_DIR = THESIS_DIR / 'data' / 'baseline' / 'data'
SPLIT_DIR = DATA_DIR / 'baseline_splits'
TEST_FILE = SPLIT_DIR / 'test_baseline.json'
OBJECT_ANNOTATION_FILE = THESIS_DIR / 'data' / 'proposed_method' / 'test.json'

RUN_MODE = 'full'  # keep aligned with the training notebook
RUN_NAME = f'bbox_indicator_visual_vi_{RUN_MODE}_lora'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / RUN_NAME

HF_MODEL_REPO_ID = 'AnhDau/bbox-indicator-visual-baseline-vi'
HF_REPO_IS_PRIVATE = True
MODEL_SOURCE = HF_MODEL_REPO_ID

LOCAL_CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints' / 'best'
RESULTS_DIR = OUTPUT_DIR / 'test_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRED_JSON = RESULTS_DIR / 'test_predictions.json'
PRED_CSV = RESULTS_DIR / 'test_predictions.csv'
METRICS_JSON = RESULTS_DIR / 'test_predictions_with_metrics.json'
METRICS_CSV = RESULTS_DIR / 'test_predictions_with_metrics.csv'
SUMMARY_JSON = RESULTS_DIR / 'eval_summary.json'
SUMMARY_CSV = RESULTS_DIR / 'eval_summary.csv'

INFERENCE_BATCH_SIZE = 128
REF_SIGLIP_BATCH_SIZE = 32
REF_SIGLIP_SCALE = 2.5
REF_SIGLIP_MAX_TEXT_LENGTH = 64

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        'Cannot find the project folder. Expected folder:\n'
        f'{PROJECT_DIR}\n\n'
        'Please upload the extracted bbox_indicator_baseline folder to THESIS_DIR. '
        'The folder should contain __init__.py, data.py, model.py, engine.py, '
        'checkpointing.py, inference.py, and utils.py.'
    )

if not PROJECT_DIR.is_dir():
    raise NotADirectoryError(f'PROJECT_DIR is not a directory: {PROJECT_DIR}')

required_package_files = [
    PROJECT_DIR / '__init__.py',
    PROJECT_DIR / 'data.py',
    PROJECT_DIR / 'model.py',
    PROJECT_DIR / 'inference.py',
]
missing_package_files = [str(p) for p in required_package_files if not p.exists()]
if missing_package_files:
    raise FileNotFoundError('Missing package files:\n' + '\n'.join(missing_package_files))

if str(THESIS_DIR) not in sys.path:
    sys.path.insert(0, str(THESIS_DIR))

os.chdir(THESIS_DIR)

from bbox_indicator_baseline.data import BBoxAwareImageCaptioningDataset
from bbox_indicator_baseline.inference import load_model, predict_rows

print('THESIS_DIR:', THESIS_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR:', DATA_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('Current working directory:', Path.cwd())
print('Imported package from:', PROJECT_DIR)

for path in [TEST_FILE]:
    print(path, 'exists:', path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

THESIS_DIR: /content/drive/MyDrive/[Thesis] Improved_transformer
PROJECT_DIR: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline
DATA_DIR: /content/drive/MyDrive/[Thesis] Improved_transformer/data/baseline/data
RESULTS_DIR: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_vi_full_lora/test_results
Current working directory: /content/drive/.shortcut-targets-by-id/1Oq2CScGIzBg7b2eNBCZtMOdpduaOAfoC/[Thesis] Improved_transformer
Imported package from: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline
/content/drive/MyDrive/[Thesis] Improved_transformer/data/baseline/data/baseline_splits/test_baseline.json exists: True


## 3. Extract image zip to the local runtime

In [4]:
candidate_zips = [
    DATA_DIR / 'image.zip',
    DATA_DIR / 'images.zip',
    PROJECT_DIR / 'data' / 'image.zip',
    PROJECT_DIR / 'data' / 'images.zip',
]
IMAGE_ZIP_FILE = next((p for p in candidate_zips if p.exists()), None)
if IMAGE_ZIP_FILE is None:
    raise FileNotFoundError('Cannot find image zip. Tried:\n' + '\n'.join(str(p) for p in candidate_zips))

LOCAL_DATA_DIR = Path('/content/dataset')
LOCAL_IMAGE_DIR = LOCAL_DATA_DIR / 'img'
LOCAL_ZIP_FILE = LOCAL_DATA_DIR / IMAGE_ZIP_FILE.name
IMAGE_EXT = '.jpg'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

if (not LOCAL_ZIP_FILE.exists()) or (LOCAL_ZIP_FILE.stat().st_size != IMAGE_ZIP_FILE.stat().st_size):
    shutil.copy2(IMAGE_ZIP_FILE, LOCAL_ZIP_FILE)

if not any(LOCAL_IMAGE_DIR.glob(f'*{IMAGE_EXT}')):
    with zipfile.ZipFile(LOCAL_ZIP_FILE, 'r') as zf:
        zf.extractall(LOCAL_DATA_DIR)

candidate_image_dirs = [LOCAL_DATA_DIR / 'img', LOCAL_DATA_DIR / 'images', LOCAL_DATA_DIR / 'image']
LOCAL_IMAGE_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob(f'*{IMAGE_EXT}'))), LOCAL_IMAGE_DIR)
print('LOCAL_IMAGE_DIR:', LOCAL_IMAGE_DIR)
print('num images:', sum(1 for _ in LOCAL_IMAGE_DIR.glob(f'*{IMAGE_EXT}')))


LOCAL_IMAGE_DIR: /content/dataset/img
num images: 10038


## 4. Hugging Face login for model download

In [5]:
from huggingface_hub import login, whoami

HF_TOKEN = "_"
if HF_REPO_IS_PRIVATE:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None

    if HF_TOKEN is None or not str(HF_TOKEN).strip():
        HF_TOKEN = os.environ.get('HF_TOKEN')

    if HF_TOKEN is None or not str(HF_TOKEN).strip():
        from getpass import getpass
        HF_TOKEN = getpass('Paste your Hugging Face READ token: ')

    HF_TOKEN = str(HF_TOKEN).strip()
    login(token=HF_TOKEN, add_to_git_credential=False)
    try:
        user_info = whoami(token=HF_TOKEN)
        print('Logged in to Hugging Face as:', user_info.get('name'))
    except Exception as exc:
        print('Logged in, but whoami failed:', exc)
else:
    print('Public HF repo mode; no login required.')


Paste your Hugging Face READ token: ··········
Logged in to Hugging Face as: AnhDau


## 5. Load model and test rows

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, tokenizer, image_processor, config, device = load_model(MODEL_SOURCE, device=device)
print('Loaded model from:', MODEL_SOURCE)
print('Loaded checkpoint config:')
print(json.dumps(config, indent=2)[:3000])

TARGET_LANG = config.get('target_lang', 'en')
BATCH_SIZE = INFERENCE_BATCH_SIZE

test_dataset = BBoxAwareImageCaptioningDataset(
    jsonl_file=TEST_FILE,
    image_dir=LOCAL_IMAGE_DIR,
    image_ext=IMAGE_EXT,
    target_lang=TARGET_LANG,
)
print('Test rows:', len(test_dataset.rows))
print('Sample:', test_dataset.rows[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

model.py: 0.00B [00:00, ?B/s]

data.py: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/711 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loaded model from: AnhDau/bbox-indicator-visual-baseline-vi
Loaded checkpoint config:
{
  "architecture": "bbox_indicator_visual_baseline",
  "decoder_memory": "[bbox_indicator_token, siglip_patch_tokens]",
  "run_mode": "full",
  "vision_model_name": "google/siglip2-large-patch16-256",
  "text_model_name": "facebook/mbart-large-50",
  "target_lang": "vi",
  "tgt_lang_code": "vi_VN",
  "bbox_indicator_ref": 1000.0,
  "bbox_indicator_dim": 4,
  "max_target_length": 64,
  "max_new_tokens": 30,
  "num_beams": 4,
  "repetition_penalty": 1.1,
  "no_repeat_ngram_size": 3,
  "length_penalty": 1.0,
  "train_batch_size": 64,
  "eval_batch_size": 64,
  "grad_accum_steps": 1,
  "effective_batch_size": 64,
  "num_epochs": 30,
  "learning_rate": 3e-05,
  "warmup_steps": 350,
  "max_grad_norm": 1.0,
  "weight_decay": 0.01,
  "label_smoothing": 0.1,
  "freeze_vision": true,
  "use_lora": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "lora_target_modules": [
    "q_proj",
    "v_

## 6. Run inference on the test split

In [7]:
pred_df = predict_rows(
    model=model,
    tokenizer=tokenizer,
    image_processor=image_processor,
    rows=test_dataset.rows,
    image_dir=LOCAL_IMAGE_DIR,
    image_ext=IMAGE_EXT,
    config=config,
    device=device,
    batch_size=BATCH_SIZE,
)

# Normalize field names for metric calculation and export.
pred_df['gt_vi'] = pred_df['ground_truth']
pred_df['bbox'] = pred_df['raw_bbox']
pred_df['image_id'] = pred_df['image_id'].astype(str)
pred_df['region_id'] = pred_df['region_id'].astype(str)

pred_df.to_csv(PRED_CSV, index=False, encoding='utf-8-sig')
records = pred_df.to_dict(orient='records')
with open(PRED_JSON, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print('Saved:', PRED_CSV)
print('Saved:', PRED_JSON)
print('Rows:', len(pred_df))
display(pred_df.head())

Saved: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_vi_full_lora/test_results/test_predictions.csv
Saved: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_vi_full_lora/test_results/test_predictions.json
Rows: 36486


,image_id,region_id,raw_bbox,ground_truth,prediction,is_empty,same_as_ground_truth,error,gt_vi,bbox
0,2322119,4676336,"[261.0, 175.0, 36.0, 100.0]",Có một tấm home plate ở đây,gậy bóng chày màu đen,False,False,,Có một tấm home plate ở đây,"[261.0, 175.0, 36.0, 100.0]"
1,2322119,4676337,"[156.0, 105.0, 39.0, 77.0]",Có một cây gậy bóng chày,một người đàn ông đang đánh bóng,False,False,,Có một cây gậy bóng chày,"[156.0, 105.0, 39.0, 77.0]"
2,2322119,4676338,"[244.0, 38.0, 18.0, 24.0]",Có một chiếc mũ bảo hiểm ở đây,Mũ bảo hiểm màu đen trên người bắt bóng.,False,False,,Có một chiếc mũ bảo hiểm ở đây,"[244.0, 38.0, 18.0, 24.0]"
3,2322119,4676339,"[121.0, 145.0, 16.0, 38.0]",Có một chiếc găng tay bắt bóng ở đây,gậy bóng chày màu đen,False,False,,Có một chiếc găng tay bắt bóng ở đây,"[121.0, 145.0, 16.0, 38.0]"
4,2322119,4676340,"[114.0, 81.0, 7.0, 32.0]",Cầu thủ bắt bóng này có mũ bảo hiểm bảo vệ,Mũ bảo hiểm màu đen trên người bắt bóng.,False,False,,Cầu thủ bắt bóng này có mũ bảo hiểm bảo vệ,"[114.0, 81.0, 7.0, 32.0]"


## 7. Load saved predictions

In [8]:
with open(PRED_JSON, 'r', encoding='utf-8') as f:
    eval_results = json.load(f)

preds = [str(r.get('prediction', '')).strip() for r in eval_results]
refs = [str(r.get('gt_vi', r.get('ground_truth', ''))).strip() for r in eval_results]
print('Loaded samples:', len(eval_results))
print('Empty prediction rate:', sum(p == '' for p in preds) / max(1, len(preds)))

Loaded samples: 36486
Empty prediction rate: 0.0


## 8. BLEU, METEOR, ROUGE-L, and CIDEr

In [9]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from pycocoevalcap.cider.cider import Cider

bleu_metric = BLEU(max_ngram_order=4, effective_order=True)
# Vietnamese should not use the English Porter stemmer.
rouge_sc = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

print('Calculating BLEU, METEOR, ROUGE-L...')
for r, pred, ref in tqdm(zip(eval_results, preds, refs), total=len(preds)):
    r['bleu4'] = round(bleu_metric.sentence_score(pred, [ref]).score, 4)
    r['meteor'] = round(nltk_meteor([ref.split()], pred.split()) * 100, 4) if pred.strip() else 0.0
    r['rougeL'] = round(rouge_sc.score(ref, pred)['rougeL'].fmeasure * 100, 4) if pred.strip() else 0.0

print('Calculating CIDEr...')
gts_dict = {i: [refs[i]] for i in range(len(refs))}
res_dict = {i: [preds[i]] for i in range(len(preds))}
try:
    _, cider_per_sample = Cider().compute_score(gts_dict, res_dict)
    for i, r in enumerate(eval_results):
        r['cider'] = round(float(cider_per_sample[i]) * 100, 4)
except Exception as exc:
    print('CIDEr failed:', repr(exc))
    for r in eval_results:
        r['cider'] = None

Calculating BLEU, METEOR, ROUGE-L...


  0%|          | 0/36486 [00:00<?, ?it/s]

Calculating CIDEr...


## 9. RefSigLIPScore-style metric with SigLIP2

The report can keep the standard metric naming convention, while this implementation uses `google/siglip2-large-patch16-256` for the visual-text similarity model. The final score uses the standard scaling factor configured by `REF_SIGLIP_SCALE`.


In [10]:
from transformers import AutoModel, AutoProcessor
from PIL import Image as PILImage

siglip_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
siglip_eval_model = AutoModel.from_pretrained(
    'google/siglip2-large-patch16-256',
    torch_dtype=siglip_dtype,
).eval().to(device)
siglip_eval_proc = AutoProcessor.from_pretrained('google/siglip2-large-patch16-256')

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [11]:
import ast

def find_image_path(image_id):
    for ext in ('.jpg', '.jpeg', '.png'):
        path = LOCAL_IMAGE_DIR / f'{image_id}{ext}'
        if path.exists():
            return path
    return None

def parse_bbox(value):
    if value is None:
        return None
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except Exception:
            return None
    if hasattr(value, 'tolist'):
        value = value.tolist()
    try:
        values = [float(v) for v in list(value)[:4]]
    except Exception:
        return None
    if len(values) != 4 or not all(np.isfinite(values)):
        return None
    return values

def safe_crop_region(image, bbox):
    bbox = parse_bbox(bbox)
    if bbox is None:
        return None
    x, y, w, h = bbox
    x1 = max(0, int(round(x)))
    y1 = max(0, int(round(y)))
    x2 = min(image.width, int(round(x + max(0, w))))
    y2 = min(image.height, int(round(y + max(0, h))))
    if x2 <= x1 or y2 <= y1:
        return None
    crop = image.crop((x1, y1, x2, y2))
    if crop.width < 2 or crop.height < 2:
        crop = crop.resize((max(2, crop.width), max(2, crop.height)))
    return crop

def set_empty_ref_siglip_scores(row):
    row['ref_siglip_visual'] = 0.0
    row['ref_siglip_text'] = 0.0
    row['ref_siglip_score_raw'] = 0.0
    row['ref_siglip_score'] = 0.0
    row['refsclipscore_siglip'] = 0.0

BATCH_SIZE_SCORE = REF_SIGLIP_BATCH_SIZE
image_cache = {}

for start in tqdm(range(0, len(eval_results), BATCH_SIZE_SCORE), desc='RefSigLIPScore'):
    batch = eval_results[start:start + BATCH_SIZE_SCORE]
    images, text_list, valid_positions = [], [], []

    for local_idx, r in enumerate(batch):
        image_id = str(r.get('image_id', ''))
        if image_id not in image_cache:
            image_path = find_image_path(image_id)
            image_cache[image_id] = PILImage.open(image_path).convert('RGB') if image_path else None

        img = image_cache[image_id]
        if img is None:
            set_empty_ref_siglip_scores(r)
            continue

        crop = safe_crop_region(img, r.get('bbox') or r.get('raw_bbox'))
        prediction_text = str(r.get('prediction', '')).strip()
        reference_text = str(r.get('gt_vi', r.get('ground_truth', ''))).strip()

        if crop is None or not prediction_text or not reference_text:
            set_empty_ref_siglip_scores(r)
            continue

        images.append(crop)
        text_list.extend([prediction_text, reference_text])
        valid_positions.append(local_idx)

    if not images:
        continue

    inputs_img = siglip_eval_proc(images=images, return_tensors='pt').to(device)
    inputs_txt = siglip_eval_proc(
        text=text_list,
        padding='max_length',
        max_length=REF_SIGLIP_MAX_TEXT_LENGTH,
        truncation=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        autocast_enabled = torch.cuda.is_available()
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
            img_feats = siglip_eval_model.get_image_features(**inputs_img)
            txt_feats = siglip_eval_model.get_text_features(**inputs_txt)
        img_feats = img_feats / img_feats.norm(p=2, dim=-1, keepdim=True).clamp_min(1e-8)
        txt_feats = txt_feats / txt_feats.norm(p=2, dim=-1, keepdim=True).clamp_min(1e-8)

    for j, local_idx in enumerate(valid_positions):
        r = batch[local_idx]
        pred_feat = txt_feats[2 * j].unsqueeze(0)
        ref_feat = txt_feats[2 * j + 1].unsqueeze(0)
        img_feat = img_feats[j].unsqueeze(0)

        visual_score = max(float((img_feat * pred_feat).sum().item()), 0.0)
        text_score = max(float((pred_feat * ref_feat).sum().item()), 0.0)
        if visual_score + text_score > 0:
            raw_ref_score = 2.0 * visual_score * text_score / (visual_score + text_score + 1e-8)
        else:
            raw_ref_score = 0.0

        scaled_score = REF_SIGLIP_SCALE * raw_ref_score * 100.0
        r['ref_siglip_visual'] = round(visual_score, 6)
        r['ref_siglip_text'] = round(text_score, 6)
        r['ref_siglip_score_raw'] = round(raw_ref_score, 6)
        r['ref_siglip_score'] = round(scaled_score, 4)
        r['refsclipscore_siglip'] = r['ref_siglip_score']

# Free evaluation model if you need memory for next cells.
del siglip_eval_model
cleanup = getattr(torch.cuda, 'empty_cache', None)
if cleanup:
    cleanup()

RefSigLIPScore:   0%|          | 0/1141 [00:00<?, ?it/s]

## 10. CHAIR-S and CHAIR-I

This section uses object annotations when they are available. If the annotation file is missing or does not contain usable object labels, CHAIR is skipped instead of producing invalid scores.


In [12]:
import nltk

# CHAIR in this notebook is based on English noun extraction and English Visual Genome object labels.
# For Vietnamese captions, this metric is not valid without a Vietnamese-to-English object mapping.
if str(TARGET_LANG).lower() == 'vi':
    print('TARGET_LANG=vi: skipping CHAIR-S/CHAIR-I because the current implementation is English-based.')
    for r in eval_results:
        r['chair_s'] = None
        r['chair_i'] = None
else:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    nltk.download('wordnet', quiet=True)
    from nltk.stem import WordNetLemmatizer

    lemmatizer = WordNetLemmatizer()

    def get_nouns(text):
        try:
            tokens = nltk.word_tokenize(str(text).lower())
            tagged = nltk.pos_tag(tokens)
            return {lemmatizer.lemmatize(w) for w, t in tagged if t.startswith('NN')}
        except Exception:
            return set()

    def get_regions_list_or_dict(item):
        regions_raw = item.get('regions_mapping', {}) if isinstance(item, dict) else {}
        if isinstance(regions_raw, dict):
            return {str(k): v for k, v in regions_raw.items()}
        if isinstance(regions_raw, list):
            return {str(i): r for i, r in enumerate(regions_raw)}
        return {}

    def find_region(item, region_id):
        regions = get_regions_list_or_dict(item)
        region_id = str(region_id)
        if region_id in regions:
            return regions[region_id]
        for key, region in regions.items():
            if isinstance(region, dict) and str(region.get('region_id', key)) == region_id:
                return region
        return {}

    def read_annotation_items(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for key in ['data', 'rows', 'samples', 'annotations', 'images']:
                value = data.get(key)
                if isinstance(value, list):
                    return value
        return []

    def build_region_objects(item, region):
        if not isinstance(item, dict):
            return set()
        annotation_blob = item.get('global_scene_' + 'graph', {}) or item.get('scene_' + 'graph', {}) or {}
        object_entries = annotation_blob.get('nodes', []) if isinstance(annotation_blob, dict) else []
        if not isinstance(object_entries, list):
            object_entries = []

        entry_lookup = {}
        for idx, entry in enumerate(object_entries):
            if not isinstance(entry, dict):
                continue
            entry_lookup[str(idx)] = entry
            if entry.get('node_id') is not None:
                entry_lookup[str(entry.get('node_id'))] = entry
            if entry.get('id') is not None:
                entry_lookup[str(entry.get('id'))] = entry

        selected_ids = region.get('node_' + 'indices', []) if isinstance(region, dict) else []
        if selected_ids:
            selected_entries = [entry_lookup[str(idx)] for idx in selected_ids if str(idx) in entry_lookup]
        else:
            selected_entries = [entry for entry in object_entries if isinstance(entry, dict)]

        objects = set()
        for entry in selected_entries:
            for field in ['canonical', 'name', 'object', 'label']:
                value = entry.get(field)
                if value:
                    for word in str(value).lower().replace(',', ' ').split():
                        objects.add(lemmatizer.lemmatize(word))
            for attr in entry.get('attributes', []) or []:
                for word in str(attr).lower().replace(',', ' ').split():
                    objects.add(lemmatizer.lemmatize(word))
        return {obj for obj in objects if obj}

    if OBJECT_ANNOTATION_FILE.exists():
        annotation_items = read_annotation_items(OBJECT_ANNOTATION_FILE)
        item_lookup = {
            str(item.get('image_id')): item
            for item in annotation_items
            if isinstance(item, dict) and item.get('image_id') is not None
        }
        print('Loaded object annotation items:', len(item_lookup))

        for r in tqdm(eval_results, desc='CHAIR'):
            item = item_lookup.get(str(r.get('image_id')))
            if item is None:
                r['chair_s'] = None
                r['chair_i'] = None
                continue

            region = find_region(item, r.get('region_id'))
            object_vocab = build_region_objects(item, region)
            pred_nouns = get_nouns(r.get('prediction', ''))

            if not pred_nouns:
                r['chair_s'] = 0.0
                r['chair_i'] = 0
            elif not object_vocab:
                r['chair_s'] = None
                r['chair_i'] = None
            else:
                hallucinated = pred_nouns - object_vocab
                r['chair_s'] = round(len(hallucinated) / len(pred_nouns), 4)
                r['chair_i'] = 1 if hallucinated else 0
    else:
        print('Object annotation file not found; skipping CHAIR.')
        for r in eval_results:
            r['chair_s'] = None
            r['chair_i'] = None

TARGET_LANG=vi: skipping CHAIR-S/CHAIR-I because the current implementation is English-based.


## 11. Save metric summary

In [13]:
def avg_metric(key):
    vals = []
    for r in eval_results:
        value = r.get(key)
        if value is None:
            continue
        try:
            value = float(value)
        except Exception:
            continue
        if np.isfinite(value):
            vals.append(value)
    return round(float(np.mean(vals)), 4) if vals else None

preds = [str(r.get('prediction', '')).strip() for r in eval_results]
refs = [str(r.get('gt_vi', r.get('ground_truth', ''))).strip() for r in eval_results]
pred_lens = [len(p.split()) for p in preds]
ref_lens = [len(r.split()) for r in refs]

avg_chair_s = avg_metric('chair_s')
avg_chair_i = avg_metric('chair_i')

summary = {
    'total_samples': len(eval_results),
    'empty_prediction_rate': round(sum(p == '' for p in preds) / max(1, len(preds)), 4),
    'unique_predictions': len(set(preds)),
    'avg_len_gt': round(float(np.mean(ref_lens)), 2) if ref_lens else 0.0,
    'avg_len_pred': round(float(np.mean(pred_lens)), 2) if pred_lens else 0.0,
    'corpus_bleu4': round(BLEU(max_ngram_order=4, effective_order=True).corpus_score(preds, [refs]).score, 4),
    'avg_sentence_bleu4': avg_metric('bleu4'),
    'avg_meteor': avg_metric('meteor'),
    'avg_rougeL': avg_metric('rougeL'),
    'avg_cider': avg_metric('cider'),
    'ref_siglip_scale': REF_SIGLIP_SCALE,
    'avg_ref_siglip_score': avg_metric('ref_siglip_score'),
    'avg_refsclipscore_siglip': avg_metric('refsclipscore_siglip'),
    'chair_s_pct': round(avg_chair_s * 100, 2) if avg_chair_s is not None else None,
    'chair_i_pct': round(avg_chair_i * 100, 2) if avg_chair_i is not None else None,
}

with open(METRICS_JSON, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

pd.DataFrame(eval_results).to_csv(METRICS_CSV, index=False, encoding='utf-8-sig')

with open(SUMMARY_JSON, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

pd.DataFrame([summary]).to_csv(SUMMARY_CSV, index=False, encoding='utf-8-sig')

print('Saved:', METRICS_JSON)
print('Saved:', METRICS_CSV)
print('Saved:', SUMMARY_JSON)
print('Saved:', SUMMARY_CSV)
for k, v in summary.items():
    print(f'{k:<28}: {v}')

Saved: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_vi_full_lora/test_results/test_predictions_with_metrics.json
Saved: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_vi_full_lora/test_results/test_predictions_with_metrics.csv
Saved: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_vi_full_lora/test_results/eval_summary.json
Saved: /content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_vi_full_lora/test_results/eval_summary.csv
total_samples               : 36486
empty_prediction_rate       : 0.0
unique_predictions          : 1265
avg_len_gt                  : 6.45
avg_len_pred                : 6.21
corpus_bleu4                : 8.0403
avg_sentence_bleu4          : 10.9481
avg_meteor                  : 20.7198
avg_rougeL                  : 37.5294
avg_cider 

## 12. Preview qualitative examples

In [14]:
preview_cols = [
    'image_id', 'region_id', 'bbox', 'gt_vi', 'prediction',
    'bleu4', 'meteor', 'rougeL', 'cider', 'ref_siglip_score', 'chair_s', 'chair_i'
]
metrics_df = pd.DataFrame(eval_results)
display(metrics_df[[c for c in preview_cols if c in metrics_df.columns]].head(20))

,image_id,region_id,bbox,gt_vi,prediction,bleu4,meteor,rougeL,cider,ref_siglip_score,chair_s,chair_i
0,2322119,4676336,"[261.0, 175.0, 36.0, 100.0]",Có một tấm home plate ở đây,gậy bóng chày màu đen,0.0000,0.0000,11.7647,0.0000,36.7758,None,None
1,2322119,4676337,"[156.0, 105.0, 39.0, 77.0]",Có một cây gậy bóng chày,một người đàn ông đang đánh bóng,7.8098,16.3934,38.0952,48.7494,24.3493,None,None
2,2322119,4676338,"[244.0, 38.0, 18.0, 24.0]",Có một chiếc mũ bảo hiểm ở đây,Mũ bảo hiểm màu đen trên người bắt bóng.,8.3922,36.3512,37.0370,104.6036,33.8608,None,None
3,2322119,4676339,"[121.0, 145.0, 16.0, 38.0]",Có một chiếc găng tay bắt bóng ở đây,gậy bóng chày màu đen,4.7998,5.8140,36.3636,27.9565,29.6138,None,None
4,2322119,4676340,"[114.0, 81.0, 7.0, 32.0]",Cầu thủ bắt bóng này có mũ bảo hiểm bảo vệ,Mũ bảo hiểm màu đen trên người bắt bóng.,10.7390,18.5185,35.2941,121.0945,21.2828,None,None
5,2322119,4676341,"[411.0, 212.0, 7.0, 21.0]",Có một đường kẻ màu trắng có thể nhìn thấy ở đây,gậy bóng chày màu đen,2.6342,4.4248,24.0000,1.8953,0.0000,None,None
6,2322119,4676342,"[23.0, 105.0, 2.0, 10.0]",Trọng tài đang mặc áo sơ mi màu xanh dương nhạt,mũ bảo hiểm màu xanh dương của người đánh bóng,14.1333,27.0133,47.0588,43.5631,31.6467,None,None
7,2322119,4676343,"[188.0, 95.0, 11.0, 31.0]",Người đàn ông này đang đeo găng tay đánh bóng,Mũ bảo hiểm màu đen trên người bắt bóng.,4.1961,5.0505,26.6667,0.0000,33.3810,None,None
8,2322119,4676346,"[251.0, 177.0, 47.0, 36.0]",home plate,gậy bóng chày màu đen,0.0000,0.0000,0.0000,0.0000,18.5486,None,None
9,2322119,4676348,"[146.0, 40.0, 140.0, 140.0]",cầu thủ chuẩn bị chạy về đích,một người đàn ông đang cầm gậy bóng chày,0.0000,0.0000,32.0000,0.0000,44.3926,None,None
